# Reinforcement Learning (RL)

A practical reference for **Reinforcement Learning** in modern AI/ML workloads — with an emphasis on the role RL plays in the LLM stack: **RLHF** (PPO), and the preference-optimization methods that grew out of it (**DPO**, **GRPO**, **RLAIF**). Covers the core RL formalism, the reward-model + policy-optimization loop, how to run it with the Hugging Face `trl` library, the infrastructure it demands, and how it compares to plain supervised fine-tuning.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**Reinforcement Learning (RL)** trains an *agent* to take *actions* in an *environment* so as to maximize a cumulative *reward* signal. Unlike supervised learning — which copies labeled answers — RL learns from **consequences**: the agent tries something, sees how good the outcome was, and shifts its behavior toward what paid off. This makes RL the natural fit whenever the "right answer" is hard to label but *quality is easy to score*.

In the MLOps / LLM world, RL is the engine behind **alignment**. A base language model is a fluent next-token predictor but has no notion of "helpful, honest, harmless." RL from Human (or AI) Feedback — **RLHF / RLAIF** — optimizes the model against a learned *reward model* that encodes human preferences, turning a raw predictor into a usable assistant. This is the third stage of the now-standard recipe: **pretrain → supervised fine-tune (SFT) → RL/preference-optimize**.

### What is it?

RL is a feedback-driven optimization loop. An agent observes a **state**, picks an **action** from its **policy** `π(a|s)`, the environment returns a **reward** and a new state, and the agent updates `π` to make high-reward trajectories more likely. For an LLM, the "state" is the prompt-so-far, an "action" is emitting the next token (or a full response), and the "reward" is a score from a reward model or a verifier (e.g. "did the unit test pass?").

### Why use it?

Key benefits of using Reinforcement Learning:

- **Optimizes for outcomes you can score but not label.** You can't write the single "best" answer to "explain quantum tunneling to a 10-year-old," but you *can* rank two answers. RL turns that comparative signal into gradients.
- **Aligns models with human preferences.** RLHF is what makes a chat model refuse harmful requests, follow instructions, and prefer helpful, well-formatted answers.
- **Goes beyond the demonstration data.** SFT can only imitate the responses it was shown; RL can discover better responses than any in the dataset, as long as the reward model recognizes them.
- **Handles sequential, long-horizon decisions** (robotics, control, game-playing, agentic tool use) where each action changes the future.

### When to use it?

Reinforcement Learning is particularly useful when:

- The objective is a **preference or a verifiable outcome** (helpfulness, safety, "tests pass," "human upvotes this") rather than a fixed label.
- You already have a competent **SFT model** and want to push quality/alignment past what imitation gives you.
- The problem is inherently **sequential** and actions have downstream consequences (control, planning, multi-step agents).
- A cheap, reliable **reward signal** exists or can be learned — RL is only as good as its reward.

## Key Features

### Core RL concepts and the methods built on them

| Feature | Description | Why it matters |
|---------|-------------|----------------|
| **Reward signal** | A scalar score for an action/trajectory (human preference, reward model, or a verifier like a test suite) | Defines *what* you're optimizing; the single most important design choice in any RL system |
| **Policy `π(a\|s)`** | The agent's (stochastic) action-selection distribution — for an LLM, the model itself | This is the thing you are training; RL updates its weights toward higher reward |
| **Value / advantage** | Estimate of expected future reward (`V`/`Q`); **advantage** = how much better an action was than baseline | Reduces gradient variance so training is stable instead of noisy |
| **PPO** (Proximal Policy Optimization) | On-policy actor-critic that clips the policy update so it never moves too far per step | The workhorse of RLHF; stable and sample-reuse-friendly |
| **DPO** (Direct Preference Optimization) | Reframes RLHF as a *supervised* classification loss on preference pairs — no reward model, no sampling | Far simpler & cheaper than PPO; the popular default for preference tuning |
| **GRPO** (Group Relative Policy Optimization) | Samples a *group* of answers per prompt and uses their mean reward as the baseline — drops the value network | Memory-efficient PPO variant; powers reasoning models (e.g. DeepSeek-R1) |
| **KL penalty / reference model** | Penalizes drift from the frozen SFT model | Stops the policy "reward-hacking" into degenerate text the reward model overrates |

Practical guidance: reach for **DPO** first (simple, stable), use **PPO/GRPO** when you have an online reward signal (a reward model or a verifier) and need the policy to explore beyond the preference dataset.

## Architecture Overview

The canonical RLHF loop (PPO flavor) wires together **four** model roles around a rollout-then-update cycle:

```
                    prompt
                      │
                      ▼
            ┌───────────────────┐   sampled response (rollout)
            │   POLICY model π  ├───────────────┐
            │   (being trained) │               │
            └─────────┬─────────┘               ▼
                      │              ┌────────────────────────┐
         per-token    │              │     REWARD model       │ → scalar reward r
         logprobs     │              │ (frozen, learned from  │
                      │              │   human preferences)   │
                      ▼              └───────────┬────────────┘
            ┌───────────────────┐                │
            │  REFERENCE model  │  KL(π‖π_ref)    │
            │  (frozen SFT)     ├──── penalty ────┤
            └───────────────────┘                ▼
                      │              reward' = r − β·KL(π‖π_ref)
                      ▼                          │
            ┌───────────────────┐                ▼
            │   VALUE / critic  │──► advantage ──► PPO clipped policy-gradient update ──► π
            └───────────────────┘
```

Each step: (1) the **policy** generates responses to a batch of prompts, (2) the **reward model** scores them, (3) a **KL penalty** against the frozen **reference** keeps the policy from drifting into gibberish, (4) the **value/critic** network estimates a baseline so you optimize the *advantage*, and (5) PPO applies a clipped policy-gradient step. Repeat.

### Components

1. **Policy model (actor).** The LLM you are actually training; it must hold gradients + optimizer state.
2. **Reference model.** A frozen copy of the SFT model. Provides the KL anchor so the policy stays close to coherent language. (PEFT/LoRA can avoid a second copy by toggling the adapter off to recover the reference.)
3. **Reward model (RM).** Usually the SFT model with a scalar regression head, trained on human preference pairs to output "how good is this response." In **RLVR** (RL with verifiable rewards) the RM is replaced by a deterministic checker — a unit test, a math grader, a regex.
4. **Value network (critic).** Estimates expected future reward to compute advantages (PPO). **GRPO and DPO remove this** — GRPO uses the group mean as the baseline; DPO has no rollouts at all.
5. **Rollout + optimization engine.** The sampler that generates responses (often a fast inference engine like vLLM) and the trainer that applies the gradient step (e.g. `trl`'s `PPOTrainer` / `GRPOTrainer` / `DPOTrainer`).

## Installation

### Prerequisites

- Python 3.9+
- PyTorch with CUDA for real training (the toy NumPy/tabular demos below run on CPU)
- An NVIDIA GPU for LLM RLHF — PPO holds up to four model copies in memory, so 24 GB+ (or LoRA/QLoRA) is realistic for 7B-class models
- A **reward signal**: a trained reward model, a preference dataset (for DPO), or a programmatic verifier (for GRPO/RLVR)

### Installation Steps

**Note**: Uncomment the following cell to install the Hugging Face RL-fine-tuning stack (`trl`).

In [ ]:
# Uncomment to install the Hugging Face RLHF / preference-optimization stack.
# %pip install -U trl transformers peft accelerate datasets
# For classic-control RL experiments (CartPole, etc.):
# %pip install -U gymnasium
# Fast rollouts for PPO/GRPO at scale:
# %pip install -U vllm

## Basic Usage

### Quick Start Example

RL's core loop is small enough to see end-to-end without any deep-learning framework. Below, a tabular **Q-learning** agent solves a tiny grid world: states are cells, actions move it around, and reward is +1 only at the goal. The agent starts knowing nothing and bootstraps a value table from experience via the temporal-difference update

```
Q(s,a) ← Q(s,a) + α · [ r + γ · max_a' Q(s',a') − Q(s,a) ]
```

This is the same *learn-from-consequences* idea that, scaled up with neural policies, becomes RLHF.

In [ ]:
# Tabular Q-learning on a 1-D corridor — pure Python/NumPy, runs anywhere.
import numpy as np

rng = np.random.default_rng(0)

N_STATES = 6          # positions 0..5; 5 is the goal
ACTIONS = (0, 1)      # 0 = left, 1 = right
GOAL = N_STATES - 1
alpha, gamma, epsilon = 0.1, 0.9, 0.2

Q = np.zeros((N_STATES, len(ACTIONS)))

def step(s, a):
    """Environment dynamics: return (next_state, reward, done)."""
    s2 = max(0, s - 1) if a == 0 else min(GOAL, s + 1)
    return s2, (1.0 if s2 == GOAL else 0.0), s2 == GOAL

for episode in range(3000):
    s = int(rng.integers(GOAL))               # random non-goal start so every state is explored
    for _ in range(50):                       # cap episode length
        # epsilon-greedy action selection (explore vs exploit)
        a = rng.integers(2) if rng.random() < epsilon else int(np.argmax(Q[s]))
        s2, r, done = step(s, a)
        # temporal-difference update toward the bootstrapped target
        Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
        s = s2
        if done:
            break

greedy = ["right" if np.argmax(Q[s]) == 1 else "left" for s in range(GOAL)]
print("Learned policy per state:", greedy)
print("Q-values:\n", np.round(Q, 2))
# The agent learns to always move right — the shortest path to the reward.

In [ ]:
# The LLM analogue: preference optimization with Hugging Face TRL.
# Gated behind try/except so the notebook runs even without the libraries installed.
try:
    from trl import DPOConfig, DPOTrainer

    # DPO needs only a preference dataset of (prompt, chosen, rejected) triples —
    # no reward model, no sampling loop. The trainer turns it into a supervised loss.
    cfg = DPOConfig(
        beta=0.1,                  # KL strength: how tightly to stay near the reference
        learning_rate=5e-6,
        per_device_train_batch_size=2,
        max_length=512,
        output_dir="/tmp/dpo-demo",
    )
    print("DPOConfig ready — beta:", cfg.beta, "| lr:", cfg.learning_rate)
    print("Usage: DPOTrainer(model, ref_model=None, args=cfg, train_dataset=pref_ds, ...).train()")
except Exception as e:  # noqa: BLE001 - libraries optional in this environment
    print("trl not installed — showing API shape only:", type(e).__name__)

## Advanced Features

### Beyond vanilla policy gradients

#### PPO: the clipped update that made RLHF stable

Naive policy gradients are high-variance and can take a catastrophic step that collapses the policy. **PPO** fixes this by maximizing a *clipped* surrogate objective. With the probability ratio `ρ = π_new(a|s) / π_old(a|s)` and advantage `Â`:

```
L_PPO = E[ min( ρ·Â ,  clip(ρ, 1−ε, 1+ε)·Â ) ]
```

The `clip` (typically `ε≈0.2`) removes the incentive to move the policy too far in one update, giving stable, sample-efficient learning. RLHF adds a **per-token KL penalty** against the reference model to the reward so the policy can't drift into text the reward model overrates but humans hate.

#### DPO: skip the reward model entirely

**Direct Preference Optimization** proves that the RLHF objective has a closed-form optimum, letting you optimize the policy *directly* on preference pairs with a simple classification-style loss:

```
L_DPO = −log σ( β · [ (log π_θ(y_w|x) − log π_ref(y_w|x))
                     −(log π_θ(y_l|x) − log π_ref(y_l|x)) ] )
```

where `y_w` is the chosen and `y_l` the rejected response. No reward model, no sampling, no value network — just two forward passes per pair. This is why DPO is the default starting point today. Variants: **IPO** (fixes DPO's overfitting), **KTO** (uses unpaired good/bad labels), **ORPO** (folds preference into SFT, no reference model).

#### GRPO: PPO without the critic

**Group Relative Policy Optimization** samples a *group* of `G` responses per prompt, scores each, and uses the group's mean reward as the baseline — so the advantage is just `(r_i − mean(r)) / std(r)`. This eliminates the value network (halving model copies and memory) and pairs naturally with **verifiable rewards** (RLVR): math/code answers graded by an exact checker. It's the method behind recent open reasoning models.

In [ ]:
# GRPO with a programmatic (verifiable) reward — no reward model needed.
# The reward function here rewards responses that are short and end with a digit;
# in practice it would run a unit test, a math grader, or a format checker.
try:
    from trl import GRPOConfig, GRPOTrainer

    def reward_len_and_digit(completions, **kwargs):
        """Return one scalar reward per sampled completion."""
        rewards = []
        for c in completions:
            text = c if isinstance(c, str) else c[-1]["content"]
            r = 1.0 if text.strip()[-1:].isdigit() else 0.0    # verifiable: ends in a number
            r -= 0.001 * len(text)                             # mild brevity pressure
            rewards.append(r)
        return rewards

    cfg = GRPOConfig(
        num_generations=8,            # group size G — the baseline is this group's mean
        learning_rate=1e-6,
        per_device_train_batch_size=8,
        max_completion_length=256,
        output_dir="/tmp/grpo-demo",
    )
    print("GRPOConfig ready — group size:", cfg.num_generations)
    print("reward_funcs accepts any callable(completions) -> list[float]; e.g. test-suite pass/fail.")
    # GRPOTrainer(model, reward_funcs=reward_len_and_digit, args=cfg, train_dataset=prompts).train()
except Exception as e:  # noqa: BLE001
    print("trl not installed — GRPO uses a group-relative baseline instead of a value net:",
          type(e).__name__)

## Use Cases

### Real-world applications of Reinforcement Learning

#### Use Case 1: Aligning a chat assistant (RLHF / RLAIF)

- **Context:** An SFT model follows instructions but is verbose, occasionally unsafe, and ignores formatting preferences. You have humans (or a stronger "judge" model, for RLAIF) who can rank pairs of responses.
- **Implementation:** Collect preference pairs → train a reward model (or skip it with DPO) → run PPO/DPO against it with a KL anchor to the SFT model. Evaluate on held-out prompts with win-rate vs the SFT baseline.
- **Results:** Higher helpfulness/safety win-rates and better instruction adherence than SFT alone — the standard last step in shipping a production chat model.

#### Use Case 2: Reasoning models with verifiable rewards (RLVR / GRPO)

- **Context:** You want a model that's better at math and coding, where correctness is *checkable* — the answer is right or wrong, the tests pass or fail.
- **Implementation:** GRPO — sample a group of solutions per problem, reward = exact-match / test-pass, no reward model needed. The group-relative baseline keeps it memory-cheap; fast rollouts via vLLM keep it fast.
- **Results:** Emergent long chain-of-thought and large accuracy gains on math/code benchmarks (the recipe behind DeepSeek-R1-style reasoning models).

## Best Practices

### Recommended practices for Reinforcement Learning

1. **Always start from a solid SFT checkpoint.** RL refines behavior; it can't teach the format/skills from scratch. Garbage-in SFT → unstable RL.
2. **Begin with DPO before reaching for PPO/GRPO.** It's simpler, has no reward model to train, no sampling loop, and is far more stable. Escalate to online RL only when you need exploration beyond the preference set or have a live verifier.
3. **Keep the KL penalty (`β`) honest.** Too low → the policy reward-hacks into degenerate text; too high → it never improves. Monitor KL-to-reference as a first-class metric and tune `β` to keep it in a sane band.
4. **Make the reward signal trustworthy.** Audit the reward model for spurious correlations (length, formatting, sycophancy). For verifiable tasks, prefer a deterministic checker over a learned RM.
5. **Use LoRA/QLoRA to fit the model zoo.** PPO juggles up to four models; adapters let one frozen base serve as policy+reference (toggle the adapter) and slash memory.
6. **Evaluate with held-out, preference-based metrics** (win-rate vs baseline, not just reward). Rising reward with falling human win-rate is the classic sign of reward hacking.

## Common Pitfalls

### What to avoid when using Reinforcement Learning

1. **Reward hacking / over-optimization.** The policy finds responses the reward model loves but humans don't (excessive length, flattery, keyword stuffing). *Avoid by* capping KL drift, using held-out human/judge eval, and regularizing or ensembling the reward model.
2. **Reward-model misspecification.** A reward model trained on biased or shallow preferences bakes those biases in and amplifies them. *Avoid by* de-biasing preference data (e.g. control for length) and spot-checking what the RM actually rewards.
3. **KL collapse or KL explosion.** Forgetting/mis-scaling the KL term lets the policy either never move or diverge into gibberish. *Avoid by* logging KL every step and using an adaptive KL controller.
4. **Training instability from a bad value network (PPO).** A poorly-fit critic produces noisy advantages and divergence. *Avoid by* warming up the value head, clipping value loss, or switching to GRPO/DPO which drop the critic.
5. **Tiny / non-representative prompt distribution.** RL only optimizes the prompts you roll out on; a narrow set yields a narrowly-improved (and elsewhere-regressed) model. *Avoid by* sampling a broad, production-like prompt mix and tracking regression on off-distribution evals.

## Performance Optimization

### Optimizing Reinforcement Learning for production

#### Where the time and memory go

PPO-style RLHF is expensive because each step does **generation then training**, and holds multiple model copies:

```
Memory (PPO, full fine-tune, 7B):
  policy (weights+grads+Adam)   ~84 GB
  reference (frozen)            ~14 GB
  reward model (frozen)         ~14 GB
  value network                 ~14 GB   → easily 100 GB+ before activations
```

Most wall-clock time is **rollout generation**, not the gradient step — sampling long responses token-by-token dominates.

#### Key parameters to optimize

- **Use a fast rollout engine.** Generate with vLLM/TGI (paged-attention, continuous batching) instead of naive `model.generate` — often the single biggest speedup.
- **Drop model copies.** GRPO removes the value net; DPO removes rollouts, the reward model, *and* the value net. LoRA lets policy and reference share one base.
- **Tune the rollout/update ratio.** Generation length, `num_generations` (GRPO group size), and PPO epochs-per-batch trade reward signal for throughput.
- **Quantize the frozen models.** The reference and reward models never train — serve them in 8-bit/4-bit to reclaim memory for the policy.

In [ ]:
# Estimate RLHF training memory for each method, by model size.
def rlhf_mem_gb(num_params_b, method="ppo", lora=False):
    P = num_params_b * 1e9
    train_per_param = 12 if not lora else 2 + 0.003 * 10   # full: w+g+adam(12); LoRA: frozen base + tiny adapter
    frozen_per_param = 2                                   # fp16 frozen copy
    policy = P * train_per_param
    if method == "ppo":
        copies = policy + P * frozen_per_param * 2 + P * train_per_param  # +ref +RM +value(trained)
    elif method == "grpo":
        copies = policy + P * frozen_per_param * 2                        # +ref +RM, no value net
    elif method == "dpo":
        copies = policy + (0 if lora else P * frozen_per_param)           # +ref (LoRA shares base via adapter toggle)
    else:
        raise ValueError(method)
    return copies / 1e9

for size in (7, 13):
    for m in ("ppo", "grpo", "dpo"):
        full = rlhf_mem_gb(size, m, lora=False)
        lo = rlhf_mem_gb(size, m, lora=True)
        print(f"{size:>2}B {m.upper():>4}  full={full:6.1f} GB   LoRA={lo:6.1f} GB")
    print()

## Production Deployment

### Deploying Reinforcement Learning in production

RL training is offline and batch-oriented; you **deploy the resulting policy** (an ordinary fine-tuned model) the same way you'd serve any LLM. The RL-specific machinery — reward model, reference, rollouts — lives only in the training pipeline. Two patterns:

1. **Offline preference tuning (DPO/GRPO/PPO) → ship the merged model.** The common case: run the RL job on a GPU cluster, evaluate, then serve the resulting weights via vLLM/TGI. The reward model is discarded at serve time.
2. **Online / continual RLHF.** A feedback loop where production thumbs-up/down data flows back into periodic preference-tuning jobs. Here the *pipeline* is the product: data collection → RM refresh → RL job → eval gate → canary rollout.

#### Docker Deployment

```dockerfile
# Training image for an RLHF / preference-tuning job.
FROM nvidia/cuda:12.4.1-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y python3-pip git && rm -rf /var/lib/apt/lists/*
RUN pip3 install --no-cache-dir trl transformers peft accelerate datasets vllm
COPY train_dpo.py /app/train_dpo.py
WORKDIR /app
# Reward signal + base model are mounted at runtime; outputs go to a model registry.
ENTRYPOINT ["accelerate", "launch", "train_dpo.py"]
```

#### Kubernetes Deployment

```yaml
# Run the RL fine-tuning as a batch Job, not a long-lived service.
apiVersion: batch/v1
kind: Job
metadata:
  name: rlhf-dpo-run
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: trainer
          image: registry.example.com/rlhf-trainer:1.0.0
          args: ["--base", "/models/sft", "--prefs", "/data/prefs.jsonl",
                 "--beta", "0.1", "--output", "/models/dpo-out"]
          resources:
            limits:
              nvidia.com/gpu: 8          # multi-GPU for the policy + frozen copies
          volumeMounts:
            - {name: models, mountPath: /models}
            - {name: data, mountPath: /data}
      volumes:
        - {name: models, persistentVolumeClaim: {claimName: model-store}}
        - {name: data, persistentVolumeClaim: {claimName: pref-data}}
```

## Monitoring and Observability

### Monitoring Reinforcement Learning in production

#### Key metrics to track

During training:
- **Mean reward / reward distribution** per batch — should rise, but *with* the quality metrics below, not instead of them.
- **KL divergence to the reference** — the canary for reward hacking. A sudden climb means the policy is fleeing coherent language.
- **Response length & entropy** — length creep is the #1 reward-hacking tell; collapsing entropy means the policy is going deterministic/degenerate.
- **Held-out win-rate / verifier pass-rate** — the metric you actually care about, on data not used for the reward.

After deployment (online RLHF):
- **Human feedback rate (thumbs up/down), task success, escalation/refusal rates** — closing the loop back into the next preference dataset.

#### Logging best practices

- Log **reward, KL, length, and entropy together** every step — they're only interpretable as a set (rising reward + rising KL + rising length = hacking).
- Snapshot **sample completions** periodically so a human can eyeball *what* the reward is steering toward.
- Version and log the **base model id, reward-model id, and `β`/hyperparameters** with every run — RL results are notoriously seed- and config-sensitive, so reproducibility hinges on this.
- Use appropriate log levels: per-step scalars to your tracker (W&B/TensorBoard) at INFO, full rollouts at DEBUG, divergence/NaN guards at WARN/ERROR.

## Troubleshooting

### Common issues with Reinforcement Learning

#### Issue 1: Reward goes up but outputs get worse

**Symptoms:** Mean reward climbs steadily while human/judge win-rate stalls or drops; responses grow long, repetitive, or sycophantic.

**Cause:** Reward hacking — the policy is exploiting flaws in the reward model rather than genuinely improving.

**Solution:** Increase the KL penalty `β`, add length normalization to the reward, evaluate on held-out human/judge metrics (not reward), and consider reward-model ensembling or a fresh RM trained on harder negatives.

#### Issue 2: Training diverges / loss becomes NaN

**Symptoms:** KL explodes, reward collapses, or gradients/loss go to NaN within a few hundred steps.

**Cause:** Learning rate too high, a mis-fit value network producing wild advantages, or no/too-weak KL anchor.

**Solution:** Lower the LR (RLHF LRs are tiny, ~1e-6), enable advantage/value clipping, add or strengthen the KL term with an adaptive controller, and warm up the value head — or switch to GRPO/DPO to remove the critic entirely.

#### Issue 3: The policy barely changes from the SFT model

**Symptoms:** Reward is flat and outputs are indistinguishable from the starting point.

**Cause:** KL penalty too strong, learning rate too low, or a reward model that gives near-constant scores (no gradient signal).

**Solution:** Lower `β`, verify the reward model actually discriminates good from bad on your prompts (check its score spread), and confirm rollouts are diverse enough (raise sampling temperature / group size) to give the optimizer signal.

## Comparison with Alternatives

### How Reinforcement Learning compares to other training strategies

| Dimension | RLHF (PPO) | DPO / GRPO | Supervised Fine-Tuning (SFT) | Prompting / RAG |
|-----------|-----------|------------|------------------------------|-----------------|
| Signal needed | Reward model + prompts | Preference pairs (DPO) or verifier (GRPO) | Labeled input→output demos | None (no training) |
| Can exceed the demo data | Yes (explores) | DPO: limited; GRPO: yes | No (imitation only) | No |
| Training complexity | High (4 models, rollouts) | Low–medium | Low | None |
| Stability | Trickiest | Much more stable | Very stable | N/A |
| Memory / compute | Highest | Medium | Low | None |
| Typical role | Final alignment step | Default preference tuning / reasoning | Stage before RL | Augment an already-capable model |

### When to choose Reinforcement Learning

Choose RL (preference optimization) when:

- You've already done SFT and need to push **alignment/quality** past what imitation gives.
- The objective is a **preference or verifiable outcome** you can score but not label.
- You can afford a trustworthy reward signal and the extra training complexity.

Prefer **SFT** when you have clean labeled demonstrations and just need the model to imitate them; prefer **prompting/RAG** when the base model is already capable and you only need to supply facts or steer format — no training required. In practice the stack is *complementary*: **SFT → DPO/GRPO/PPO**, with RAG layered on at serving time. See the companion notebooks on Supervised Fine-Tuning and Instruction Fine-Tuning.

## Resources

### Official Documentation

- Hugging Face TRL (RLHF/DPO/GRPO/PPO) docs: https://huggingface.co/docs/trl
- TRL GitHub repository: https://github.com/huggingface/trl
- Gymnasium (classic RL environments): https://gymnasium.farama.org/

### Papers

- Proximal Policy Optimization (PPO) — https://arxiv.org/abs/1707.06347
- InstructGPT: Training LMs to follow instructions with human feedback (RLHF) — https://arxiv.org/abs/2203.02155
- Direct Preference Optimization (DPO) — https://arxiv.org/abs/2305.18290
- DeepSeekMath / GRPO — https://arxiv.org/abs/2402.03300
- Constitutional AI (RLAIF) — https://arxiv.org/abs/2212.08073

### Tutorials and Guides

- Hugging Face RLHF blog (Illustrating Reinforcement Learning from Human Feedback) — https://huggingface.co/blog/rlhf
- TRL DPO trainer guide — https://huggingface.co/docs/trl/dpo_trainer
- Spinning Up in Deep RL (OpenAI) — https://spinningup.openai.com/

### Community Resources

- Hugging Face forums — https://discuss.huggingface.co/
- r/reinforcementlearning — https://www.reddit.com/r/reinforcementlearning/
- Stack Overflow tag — https://stackoverflow.com/questions/tagged/reinforcement-learning

### Related Technologies

- Supervised / Instruction Fine-Tuning (companion notebooks — the stage before RL)
- Parameter-Efficient Fine-Tuning (LoRA/QLoRA — how RLHF fits in memory)
- Reward modeling and LLM-as-a-judge evaluation